# Building the Capacity Allocator, one constraint family at a time

Every cell below pairs three things: the **formulation** (the algebra), the **code** that implements it, and the **effect** that code had
on the model - how many variables and constraints it added.

The point is that a symbol can be located. If the question is "where is the backlog band?",
the answer is the cell titled *The outbound backlog band*: the algebra, `_add_ob_band`, and
the 92 variables and 110 constraints it contributes.

This notebook does everything `run_demo.py` does - generate inputs, build, solve twice,
publish - but with the build visible. `run_demo.py` is the same path with the stages
collapsed into one call.

**Companion notebooks.** `02-solve-the-ladder.ipynb` takes over where this one solves.
`../presentation.ipynb` charts the results from the published CSVs only.

In [20]:
import inspect
import sys
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import Code, Markdown, display

# Which interpreter is this? The stock kernelspec carries a bare `python` in its argv, so the
# kernel runs whatever was on PATH when the Jupyter server started - not necessarily the one
# the package was developed against. Printed rather than asserted.
print(f"kernel: Python {sys.version.split()[0]}")

HERE = Path.cwd()
MODEL = HERE.parent.parent / "model"
assert (MODEL / "capacity_allocator").is_dir(), (
    f"run this from output/notebooks/, not {HERE}"
)
sys.path.insert(0, str(MODEL))

from capacity_allocator import (
    config as cfg, data, model, run, scenarios,
)

#: Where this notebook publishes. A scratch directory by DEFAULT and deliberately: running
#: a top-to-bottom run should never overwrite the published CSVs in ../../output. Point it
#: there only when you mean to.
OUT = Path(tempfile.mkdtemp(prefix="capacity-allocator-nb-"))
print(f"publishing to: {OUT.name}")

kernel: Python 3.12.10
publishing to: capacity-allocator-nb-cj7o3orw


In [21]:
# --- the two helpers every stage cell uses ---------------------------------

def show_source(fn):
    """Render a function's source, syntax-highlighted, with where it lives."""
    src = inspect.getsource(fn)
    where = f"{Path(inspect.getsourcefile(fn)).name}:{inspect.getsourcelines(fn)[1]}"
    display(Markdown(f"`{fn.__qualname__}` &nbsp;&mdash;&nbsp; `{where}`, "
                     f"{len(src.splitlines())} lines"))
    display(Code(src, language="python"))


_SEEN = {"vars": 0, "cons": 0}

def report(m, label=""):
    """Print the model size, and what the last stage added to it.

    `report_size()` asks the solver, not the Python objects, so this is the real count
    after any presolve-time expansion of the general constraints.
    """
    nv, nc = m.backend.report_size()
    dv, dc = nv - _SEEN["vars"], nc - _SEEN["cons"]
    _SEEN.update(vars=nv, cons=nc)
    print(f"{label or 'cumulative':<20} {nv:>4} vars  {nc:>4} cons"
          f"   (+{dv} vars, +{dc} cons)")


def reset_counters():
    _SEEN.update(vars=0, cons=0)

---
## 1. The geometry: sets and indices

Nothing here is chosen by the model; it is the shape of the site and the
day.

| Symbol | Meaning |
|---|---|
| $d \in \mathcal{D}$ | Departments: Outbound, Inbound, Returns, Support. Support produces nothing and is a destination only. |
| $\mathcal{D}^{+}$ | The producing departments, $\mathcal{D}$ without Support. Only these carry capacity, backlog and limits. |
| $h \in \mathcal{H}$ | The 24 hourly steps of the horizon, 06:00 to 06:00. |
| $\mathcal{H}^{\mathrm{w}}$ | Workable hours: those fully covered by a work block. Breaks and closures are excluded, not averaged in as zero capacity. |
| $q \in \mathcal{Q}$ | Work blocks - the decision points. A break splits a shift into two blocks, because headcount can change at it. |
| $S \in \mathcal{S}$ | Skill profiles. A profile is a **set** of departments and always contains its holder's home department, so staying put is always available. |
| $c \in \mathcal{C}$ | Carrier cut-offs, in time order. |
| $\mathcal{H}_{0}, \mathcal{H}_{1}$ | Hours in the current shift, and hours after it. Cut-offs split the same way. **This split is what lets the ladder rank today above tomorrow.** |
| $\mathcal{W}_{h}$ | Band window: $h$ plus the next two *workable* hours. |
| $\mathcal{R}_{h}$ | Returns window: the workable hours from $h$ onward. |

In [22]:
print(f"departments      D  = {list(cfg.DEPTS)}")
print(f"producing        D+ = {[d for d in cfg.DEPTS if d != cfg.SP]}")
print(f"horizon          |H| = {cfg.HORIZON_HOURS} hours, end-of-day hour = {cfg.EOD_HOUR}")
print(f"support dept     SP = {cfg.SP!r}  (destination only, nobody rostered onto it)")

departments      D  = ['OB', 'IB', 'RT', 'SP']
producing        D+ = ['OB', 'IB', 'RT']
horizon          |H| = 24 hours, end-of-day hour = 23
support dept     SP = 'SP'  (destination only, nobody rostered onto it)


---
## 2. The inputs

Everything the model is *told* is listed here. All of it is constant at solve time.
`data.generate` writes the eight input frames and `data.build` turns them into a
`Params` object - that function is the model's entire input boundary.

In [23]:
scn = scenarios.get("baseline")
print(f"scenario: {scn.name} - {scn.headline}\n")

frames = data.generate(scn)
for name, f in frames.items():
    print(f"  {name:<34} {f.shape[0]:>4} rows x {f.shape[1]} cols")

p = data.build(frames)
print(f"\nParams: {len(p.quarters)} work blocks, {len(p.hours)} hours, "
      f"{len(p.cpts)} cut-offs, {len(p.pool)} non-empty skill groups")

scenario: baseline - Nothing is wrong, and the plan is still not optimal

  shifts                                9 rows x 6 cols
  headcount                            66 rows x 7 cols
  tph                                  72 rows x 4 cols
  ob_plan                              24 rows x 9 cols
  ib_plan                              24 rows x 6 cols
  returns_plan                         24 rows x 5 cols
  cpts                                  6 rows x 4 cols
  eod_guidance                          1 rows x 5 cols

Params: 6 work blocks, 24 hours, 6 cut-offs, 66 non-empty skill groups


In [24]:
# The headcount pool, which is what the people stage iterates: (block, home dept, profile).
pool = pd.DataFrame(
    [{"block": q, "home": i, "profile": "+".join(S), "people": n}
     for (q, i, S), n in p.pool.items() if n > 0]
)
print(f"{len(pool)} skill groups carrying people, {int(pool['people'].sum())} people total")
pool.head(12)

66 skill groups carrying people, 696 people total


,block,home,profile,people
0,1,OB,OB,41
1,1,OB,IB+OB,17
2,1,OB,IB+OB+SP,11
3,1,OB,OB+SP,8
4,1,IB,IB,13
5,1,IB,IB+OB,11
6,1,IB,IB+OB+SP,7
7,1,IB,IB+RT,4
8,1,RT,RT,5
9,1,RT,OB+RT,4


---
## 3. The decision variables, and where each lives in the code

Only the four people variables are genuinely free - every flow quantity
below them is implied.

The `code` column is the point of this cell. It is **asserted against the `Model` dataclass**
below, so renaming an attribute without updating the table fails here rather than quietly
leaving the table wrong.

In [25]:
SYMBOLS = [
    # (symbol, meaning, attribute on Model)
    ("y_{q,i,S->j}",      "people of profile S moved from i to j at block q",   "y"),
    ("H_{q,d,S}",         "people of profile S working in d during q",          "H"),
    ("P_{q,d}",           "headcount in d during q",                            "P"),
    ("X_{q,i->j}",        "reported movements (derived, never free)",            "X"),
    ("Hcap_{d,h}",        "capacity, in units",                                 "Hcap"),
    ("V^raw_{d,h}",       "raw availability U+A - the only variable free in sign", "V_raw"),
    ("V_{d,h}",           "availability: V^raw floored at zero",                "V"),
    ("Y_{d,h}",           "throughput",                                          "Y"),
    ("U_{d,h}",           "backlog at the start of the hour",                    "U"),
    ("U^tot_{h}",         "total outbound units, the end-of-day numerator",      "U_total"),
    ("S^bl_{h}",          "capacity summed over the band window",                "S_BL"),
    ("C^rt_{h}",          "capacity summed over the returns window",             "C_RT"),
    ("H^cum_{c}",         "capacity summed over all hours up to cut-off c",      "H_cum"),
    ("o_{h}",             "SLACK: inbound backlog over the storage ceiling",     "o_storage"),
    ("v^up_{h}",          "SLACK: outbound backlog above the band",              "v_plus"),
    ("v^down_{h}",        "SLACK: outbound backlog below the band",              "v_minus"),
    ("e^max_eod",         "SLACK: end-of-day total above target (scalar)",       "e_eod_max"),
    ("e^min_eod",         "SLACK: end-of-day total below target (scalar)",       "e_eod_min"),
    ("g_{c}",             "SLACK: demand at cut-off c over tolerated capacity",  "g_cpt"),
    ("w^rt_{h}",          "SLACK: returns backlog over its days-of-cover ceiling", "w_rt"),
    ("o^buf_{h}",         "GUARDRAIL: inbound backlog above the buffer line",    "o_storage_buffer"),
    ("d^ctr_{h}",         "GUARDRAIL: outbound backlog outside the centre deadband", "d_centre"),
]

fields = set(model.Model.__dataclass_fields__)
missing = [a for _, _, a in SYMBOLS if a not in fields]
assert not missing, (
    f"these are in the table but not on Model: {missing}\n"
    f"either the attribute was renamed, or the table is stale - fix the SYMBOLS list above"
)

# The reverse direction, reported rather than asserted: Model carries bookkeeping fields
# (backend, params, freeze_moves, tiers) that are not symbols in the formulation.
BOOKKEEPING = {"backend", "params", "freeze_moves", "tiers"}
unmapped = sorted(fields - {a for _, _, a in SYMBOLS} - BOOKKEEPING)
print(f"{len(SYMBOLS)} symbols, all present on Model.")
print(f"unmapped non-bookkeeping fields: {unmapped or 'none'}")

pd.DataFrame(SYMBOLS, columns=["symbol", "meaning", "code"]).set_index("symbol")

22 symbols, all present on Model.
unmapped non-bookkeeping fields: none


,meaning,code
symbol,,
"y_{q,i,S->j}",people of profile S moved from i to j at block q,y
"H_{q,d,S}",people of profile S working in d during q,H
"P_{q,d}",headcount in d during q,P
"X_{q,i->j}","reported movements (derived, never free)",X
"Hcap_{d,h}","capacity, in units",Hcap
"V^raw_{d,h}",raw availability U+A - the only variable free ...,V_raw
"V_{d,h}",availability: V^raw floored at zero,V
"Y_{d,h}",throughput,Y
"U_{d,h}",backlog at the start of the hour,U


---
## 4. Build it, one stage at a time

`model.build()` creates an empty `Model` and calls eight stages in order. The order is
exported as `model.STAGES` so this notebook and `build()` read one list rather than two.

People come first because every flow quantity is implied by them; capacity and flow before
the limits, because a limit is a constraint *on* the flow.

In [26]:
print([fn.__name__ for fn in model.STAGES])

backend = model.make_backend(label="notebook_after")
m = model.Model(backend=backend, params=p, freeze_moves=False)
reset_counters()
report(m, "empty model")

['_add_people', '_add_capacity', '_add_flow', '_add_storage', '_add_ob_band', '_add_eod', '_add_cpt_risk', '_add_returns_band']
empty model             0 vars     0 cons   (+0 vars, +0 cons)


---
### 4.1 People: the chain that makes a recommendation executable

Labour is conserved through **skill profiles** rather than headcount totals. This is the
part that makes the output executable: you do not move "people", you move people with a
specific skill profile, and every profile is tracked separately.

$$
\begin{aligned}
  \sum_{j \in S} y_{q,i,S \to j} &= n_{q,i,S}
    && \text{everyone in a skill group goes somewhere in it}\\
  H_{q,j,S} &= \sum_{i \in \mathcal{D}^{+}} y_{q,i,S \to j}
    && \text{who ends up in } j \text{ holding profile } S\\
  P_{q,d} &= \sum_{S \ni d} H_{q,d,S}
    && \text{a department's headcount is the sum of its profiles}\\
  X_{q,i \to j} &= \sum_{S} y_{q,i,S \to j}, \quad i \neq j
    && \text{reported movements are derived, never free}
\end{aligned}
$$

Because $y_{q,i,S \to j}$ **exists only for $j \in S$**, the first equation cannot send
anyone to a department they are not trained for: the variable that would carry them does not
exist. Nothing elsewhere has to re-check it, and no upper bound on $P_{q,d}$ is needed
either - labour conservation already implies one.

The *before* solve adds one constraint, $X_{q,i \to j} = 0$ for all $q,i,j$. Headcount then
collapses to the planned assignment and every recursion below evaluates to the planned
trajectory exactly. That is the **replay identity** the whole before/after comparison rests
on, and it is why `freeze_moves` lives on the model.

In [27]:
show_source(model._add_people)
model._add_people(m)
report(m, '_add_people')

`_add_people` &nbsp;&mdash;&nbsp; `model.py:253`, 69 lines

def _add_people(m: Model) -> None:
    """Labour conservation through skill profiles.

    Every profile is tracked separately, so nobody is placed in a department they are not
    trained for. `m.freeze_moves` pins all movements to zero.
    """
    b, p, xsum = m.backend, m.params, m.backend.xsum
    freeze_moves = m.freeze_moves

    for q in p.quarters:
        for d in cfg.DEPTS:
            m.P[(q, d)] = b.int_var(f"P_q{q}_{d}")

    # (1) Everybody in a skill group goes somewhere in their skill set.
    for (q, origin, skills), people in p.pool.items():
        if people <= 0:
            continue
        for j in skills:
            m.y[(q, origin, skills, j)] = b.int_var(
                f"y_q{q}_{origin}_{'_'.join(skills)}_to_{j}"
            )
        b.add(
            xsum(m.y[(q, origin, skills, j)] for j in skills) == people,
            f"split_q{q}_{origin}_{'_'.join(skills)}",
        )

    # (2) Headcount by destination and skill profile.
    #
    # `dict.fromkeys`, not a set: hash order is randomised per process, and variable order
    # changes which of several equally optimal solutions the solver returns.
    profiles = list(dict.fromkeys((q, skills) for (q, _o, skills) in p.pool))
    for q, skills in profiles:
        for j in skills:
            terms = [
                m.y[(q, o, s, j)]
                for (qq, o, s, jj) in m.y
                if qq == q and s == skills and jj == j
            ]
            if terms:
                m.H[(q, j, skills)] = b.int_var(f"H_q{q}_{j}_{'_'.join(skills)}")
                b.add(
                    m.H[(q, j, skills)] == xsum(terms),
                    f"defH_q{q}_{j}_{'_'.join(skills)}",
                )

    # (3) Department headcount is the sum over profiles present in it.
    for q in p.quarters:
        for d in cfg.DEPTS:
            terms = [v for (qq, j, _s), v in m.H.items() if qq == q and j == d]
            b.add(m.P[(q, d)] == (xsum(terms) if terms else 0), f"P_q{q}_{d}")

    # (4) Reported movements are DERIVED from the profile splits, never free variables.
    for q in p.quarters:
        for i in cfg.DEPTS:
            for j in cfg.DEPTS:
                if i == j:
                    continue
                m.X[(q, i, j)] = b.int_var(f"X_q{q}_{i}_to_{j}")
                terms = [
                    m.y[(qq, o, s, jj)]
                    for (qq, o, s, jj) in m.y
                    if qq == q and o == i and jj == j
                ]
                b.add(
                    m.X[(q, i, j)] == (xsum(terms) if terms else 0),
                    f"linkX_q{q}_{i}_to_{j}",
                )
                if freeze_moves:
                    b.add(m.X[(q, i, j)] == 0, f"nomove_q{q}_{i}_to_{j}")

_add_people           330 vars   264 cons   (+330 vars, +264 cons)


---
### 4.2 Capacity, and the ceiling people cannot lift

$$
\begin{aligned}
  \mathrm{Hcap}_{d,h} &= \mathrm{TPH}_{d,h}\sum_{q \in \mathcal{Q}} P_{q,d}\,W_{h,q}
    \;+\; W^{\mathrm{c}}_{h}\,\mathrm{plan}_{d,h},
    \quad d \in \mathcal{D}^{+}
    && \text{(capacity)}\\
  \mathrm{Hcap}_{\mathrm{OB},h} &\le M_{h}
    && \text{the machine ceiling}
\end{aligned}
$$

The first is **linear in headcount**, which is what keeps the whole model tractable: units
per person-hour times the people present, weighted by how much of the hour each block
covers. The second term borrows planned capacity for any fraction of an hour no block
covers, and is identically zero in this instance.

The machine ceiling is the **one hard limit in the model**. Every other limit below carries
a slack and can be broken at a price. This one cannot, because no reallocation of people
lifts a machine ceiling.

In [28]:
show_source(model._add_capacity)
model._add_capacity(m)
report(m, '_add_capacity')

`_add_capacity` &nbsp;&mdash;&nbsp; `model.py:327`, 25 lines

def _add_capacity(m: Model) -> None:
    """Units capacity per department per hour.

        capacity = TPH x (heads weighted by how much of the hour they cover)
                 + planned capacity for the uncovered fraction

    The edge term is zero here because every work-block boundary is a whole hour.
    """
    b, p, xsum = m.backend, m.params, m.backend.xsum

    for d in cfg.PRODUCING:
        for h in p.hours:
            m.Hcap[(d, h)] = b.num_var(f"Hcap_{d}_{int(h)}")
            heads = xsum(
                m.P[(q, d)] * p.cover[(h, q)] for q in p.quarters if (h, q) in p.cover
            )
            edge = p.edge.get(h, 0.0) * p.planned_cap.get((d, h), 0.0)
            b.add(
                m.Hcap[(d, h)] == p.tph.get((d, h), 0.0) * heads + edge,
                f"cap_{d}_{int(h)}",
            )

    # The machine ceiling.
    for h in p.hours:
        b.add(m.Hcap[(cfg.OB, h)] <= p.ob_machine[h], f"machine_OB_{int(h)}")

_add_capacity         402 vars   360 cons   (+72 vars, +96 cons)


---
### 4.3 Availability, throughput and the backlog recursion

$$
\begin{aligned}
  V^{\mathrm{raw}}_{d,h} &= U_{d,h} + A_{d,h}
    && \text{what is on hand, before flooring}\\
  V_{d,h} &= \max\bigl(V^{\mathrm{raw}}_{d,h},\,0\bigr)
    && \text{a negative pool is no pool}\\
  Y_{d,h} &= \min\bigl(\mathrm{Hcap}_{d,h},\, V_{d,h}\bigr)
    && \text{process the lesser of capacity and work}\\
  U_{d,h+1} &= V_{d,h} - Y_{d,h}
    && \text{what is left over becomes next hour's backlog}\\
  U_{d,h_{0}} &= U^{0}_{d}
    && \text{opening backlog}\\
  U^{\mathrm{tot}}_{h+1} &= U^{\mathrm{tot}}_{h} + A_{\mathrm{OB},h} - Y_{\mathrm{OB},h}
    && \text{the total carries the same throughput}
\end{aligned}
$$

These two $\max$/$\min$ relations are the model's **two non-linearities**, and they are why
this is a mixed-integer program rather than a linear one. The throughput relation is what
makes "nothing can be processed before it arrives" a *constraint* rather than an assumption.

The flooring is the subtle one, and the order matters. Arrivals $A$ are **signed**: a hold
going on, a cancellation, or the pick-to-ship timing gap all remove units from the pool
without anything being shipped - roughly 5% of hours. So $V^{\mathrm{raw}}$ has to be allowed
below zero, and it is the only variable in the model that is. Lower-bounding the *throughput*
at zero instead, which looks equivalent, makes the model report infeasible on its first rank.

In [29]:
show_source(model._add_flow)
model._add_flow(m)
report(m, '_add_flow')

`_add_flow` &nbsp;&mdash;&nbsp; `model.py:357`, 45 lines

def _add_flow(m: Model) -> None:
    """Backlog recursion per department.

        available   = backlog + arrivals
        throughput  = min(capacity, available)
        backlog[+1] = available - throughput
    """
    b, p = m.backend, m.params
    hours = p.hours

    for d in cfg.PRODUCING:
        for h in hours:
            m.U[(d, h)] = b.num_var(f"U_{d}_{int(h)}")
            # The only variable allowed below zero: arrivals are signed (holds,
            # cancellations), so `backlog + arrivals` can be negative.
            m.V_raw[(d, h)] = b.num_var(f"Vraw_{d}_{int(h)}", lb=-cfg.BIG_M)
            m.V[(d, h)] = b.num_var(f"V_{d}_{int(h)}")
            m.Y[(d, h)] = b.num_var(f"Y_{d}_{int(h)}")

        b.add(m.U[(d, hours[0])] == p.seed_backlog[d], f"seed_{d}")
        for h in hours:
            b.add(
                m.V_raw[(d, h)] == m.U[(d, h)] + p.arrivals.get((d, h), 0.0),
                f"avail_raw_{d}_{int(h)}",
            )
            b.add_max_zero(m.V[(d, h)], m.V_raw[(d, h)], f"avail_{d}_{int(h)}")
            b.add_min(
                m.Y[(d, h)], [m.Hcap[(d, h)], m.V[(d, h)]], f"wc_{d}_{int(h)}"
            )
        for prev, nxt in zip(hours[:-1], hours[1:]):
            b.add(
                m.U[(d, nxt)] == m.V[(d, prev)] - m.Y[(d, prev)],
                f"flow_{d}_{int(prev)}_{int(nxt)}",
            )

    # Total outbound units, the end-of-day numerator.
    for h in hours:
        m.U_total[h] = b.num_var(f"U_total_{int(h)}")
    b.add(m.U_total[hours[0]] == p.seed_ob_total, "seed_total")
    for prev, nxt in zip(hours[:-1], hours[1:]):
        b.add(
            m.U_total[nxt]
            == m.U_total[prev] + p.arrivals.get((cfg.OB, prev), 0.0) - m.Y[(cfg.OB, prev)],
            f"flow_total_{int(prev)}_{int(nxt)}",
        )

_add_flow             714 vars   528 cons   (+312 vars, +168 cons)


---
### 4.4 Limit 1: inbound storage

$$
\begin{aligned}
  U_{\mathrm{IB},h} &\le \rho^{\mathrm{sto}}\,K + o_{h}
    && \text{the building has a physical ceiling}
\end{aligned}
$$

This stage also adds the storage **guardrail**, a second tighter ceiling running parallel to
it at $\rho^{\mathrm{buf}} = 0.90$:

$$
\begin{aligned}
  U_{\mathrm{IB},h} &\le \rho^{\mathrm{buf}}\,K + o^{\mathrm{buf}}_{h}
    && \text{headroom, not the last pallet}
\end{aligned}
$$

Storage is one-sided, so there is no centre to aim for - which is why its guardrail is just a
tighter version of the same inequality, where the outbound one below needs a zone.

In [30]:
show_source(model._add_storage)
model._add_storage(m)
report(m, '_add_storage')

`_add_storage` &nbsp;&mdash;&nbsp; `model.py:407`, 19 lines

def _add_storage(m: Model) -> None:
    """Inbound backlog against the storage ceiling. One soft level, penalised per hour."""
    b, p = m.backend, m.params
    cap = p.ib_storage

    for h in p.hours:
        m.o_storage[h] = b.num_var(f"o_storage_{int(h)}")
        b.add(
            m.U[(cfg.IB, h)] <= cfg.THRESHOLDS["storage_limit"] * cap + m.o_storage[h],
            f"storage_{int(h)}",
        )

        # --- bottom guardrail: a buffer line under the ceiling --------------
        m.o_storage_buffer[h] = b.num_var(f"o_storage_buffer_{int(h)}")
        b.add(
            m.U[(cfg.IB, h)]
            <= cfg.THRESHOLDS["storage_buffer"] * cap + m.o_storage_buffer[h],
            f"storage_buffer_{int(h)}",
        )

_add_storage          762 vars   576 cons   (+48 vars, +48 cons)


---
### 4.5 Limit 2: the outbound backlog band, and the centre guardrail

The band is stated in **hours of forward cover**, not units, because that is the unit an
operator reads. The denominator is the capacity the site will actually have over the next
three workable hours, so it moves with the recommendation:

$$
\begin{aligned}
  S^{\mathrm{bl}}_{h} &= \sum_{k \in \mathcal{W}_{h}} \mathrm{Hcap}_{\mathrm{OB},k},
  \qquad r_{h} = S^{\mathrm{bl}}_{h} \big/ |\mathcal{W}_{h}|
    && \text{the forward rate}\\
  U_{\mathrm{OB},h} &\le \beta^{+}_{h}\, r_{h} + v^{\uparrow}_{h}
    && \text{not so high that cut-offs are at risk}\\
  U_{\mathrm{OB},h} &\ge \beta^{-}_{h}\, r_{h} - v^{\downarrow}_{h}
    && \text{not so low that people stand idle}
\end{aligned}
$$

Where the three-hour window runs off the end of the horizon the constraint is **skipped**
rather than stated against a short window. Dividing a clipped window by a literal three
understates the rate and tightens the band artificially, which would manufacture breaches in
the last hours of every day.

The centre guardrail, with $\bar\beta_{h}$ the band centre and $\bar r_{h}$ the **planned**
forward rate:

$$
\begin{aligned}
  d^{\mathrm{ctr}}_{h} &\ge U_{\mathrm{OB},h} - \bigl(\bar\beta_{h} + \Delta\bigr)\,\bar r_{h}
    && \text{how far above the zone}\\
  d^{\mathrm{ctr}}_{h} &\ge \bigl(\bar\beta_{h} - \Delta\bigr)\,\bar r_{h} - U_{\mathrm{OB},h}
    && \text{how far below it}
\end{aligned}
$$

Three choices worth naming. **It is a zone, not a point** ($\pm\Delta = 0.5$ hours): a bare
distance from the centre can never reach zero, and a term that can never be satisfied sitting
directly above movement minimisation removes the cost ceiling on movements entirely. **Two
inequalities linearise an absolute value.** And **the zone is anchored on the plan, not the
variable rate** - otherwise the model could satisfy the guardrail by *lowering* capacity until
a smaller rate made the backlog look centred.

In [31]:
show_source(model._add_ob_band)
model._add_ob_band(m)
report(m, '_add_ob_band')

`_add_ob_band` &nbsp;&mdash;&nbsp; `model.py:431`, 35 lines

def _add_ob_band(m: Model) -> None:
    """Outbound backlog inside its band, measured in hours of forward cover."""
    b, p, xsum = m.backend, m.params, m.backend.xsum

    for h in p.hours:
        window = p.bl_window[h]
        m.v_minus[h] = b.num_var(f"v_minus_{int(h)}")
        m.v_plus[h] = b.num_var(f"v_plus_{int(h)}")
        if len(window) < cfg.BL_WINDOW_HOURS:
            continue
        m.S_BL[h] = b.num_var(f"S_BL_{int(h)}")
        b.add(
            m.S_BL[h] == xsum(m.Hcap[(cfg.OB, k)] for k in window),
            f"SBL_{int(h)}",
        )
        rate = m.S_BL[h] * (1.0 / len(window))
        b.add(
            m.U[(cfg.OB, h)] <= p.bl_max_hours[h] * rate + m.v_minus[h],
            f"blmax_{int(h)}",
        )
        b.add(
            m.U[(cfg.OB, h)] >= p.bl_min_hours[h] * rate - m.v_plus[h],
            f"blmin_{int(h)}",
        )

        # --- bottom guardrail: distance outside a zone around the band centre ---
        plan_window = [p.planned_cap.get((cfg.OB, k), 0.0) for k in window]
        planned_rate = sum(plan_window) / len(window)
        centre_hours = 0.5 * (p.bl_min_hours[h] + p.bl_max_hours[h])
        dead_hours = cfg.THRESHOLDS["ob_bl_centre_deadband_hours"]
        zone_hi = (centre_hours + dead_hours) * planned_rate
        zone_lo = (centre_hours - dead_hours) * planned_rate
        m.d_centre[h] = b.num_var(f"d_centre_{int(h)}")
        b.add(m.d_centre[h] >= m.U[(cfg.OB, h)] - zone_hi, f"centre_hi_{int(h)}")
        b.add(m.d_centre[h] >= zone_lo - m.U[(cfg.OB, h)], f"centre_lo_{int(h)}")

_add_ob_band          854 vars   686 cons   (+92 vars, +110 cons)


---
### 4.6 Limit 3: the end-of-day target

One scalar constraint, not one per hour. The denominator is a fixed input, which is what
keeps it linear:

$$
\begin{aligned}
  U^{\mathrm{tot}}_{h^{\star}} &\le \eta^{+} E + e^{\max}_{\mathrm{eod}},
  \qquad
  U^{\mathrm{tot}}_{h^{\star}} \ge \eta^{-} E - e^{\min}_{\mathrm{eod}}
\end{aligned}
$$

Note how little it costs - two variables, two constraints. The point is structural rather
than algebraic: a single **day-level** bound propagates *back* into hourly decisions, which
is something an hourly dashboard cannot do.

In [32]:
show_source(model._add_eod)
model._add_eod(m)
report(m, '_add_eod')

`_add_eod` &nbsp;&mdash;&nbsp; `model.py:471`, 15 lines

def _add_eod(m: Model) -> None:
    """A single day-level bound on the total outbound backlog."""
    b, p = m.backend, m.params
    den = p.eod_capacity_per_hour

    m.e_eod_max = b.num_var("e_eod_max")
    m.e_eod_min = b.num_var("e_eod_min")
    b.add(
        m.U_total[p.eod_hour] <= p.eod_soft_max * den + m.e_eod_max,
        "eod_soft_max",
    )
    b.add(
        m.U_total[p.eod_hour] >= p.eod_hard_min * den - m.e_eod_min,
        "eod_hard_min",
    )

_add_eod              856 vars   688 cons   (+2 vars, +2 cons)


---
### 4.7 Limit 4: carrier cut-off risk

Capacity has a deadline, not just a total. Capacity produced in an hour counts toward
whichever cut-off window the hour falls in, pro rata through $\alpha$, and that is what makes
cumulative-capacity-to-a-deadline linear in headcount:

$$
\begin{aligned}
  H^{\mathrm{cum}}_{c} &= \sum_{k \le c} \sum_{h \in \mathcal{H}}
    \alpha_{k,h}\, \mathrm{Hcap}_{\mathrm{OB},h}
    && \text{capacity available before deadline } c\\
  D_{c} &\le \rho^{\mathrm{cpt}}\, H^{\mathrm{cum}}_{c} + g_{c}
    && \text{everything due by } c \text{ has to fit in it}
\end{aligned}
$$

$\rho^{\mathrm{cpt}} = 0.95$, below one, so the model leaves headroom rather than planning to
scrape in. One assumption is invisible in the algebra and worth stating: capacity is
**fungible within a window** - a unit picked at the start is as good as one picked at the end,
and nothing models the order cut-offs are actually worked in.

In [33]:
show_source(model._add_cpt_risk)
model._add_cpt_risk(m)
report(m, '_add_cpt_risk')

`_add_cpt_risk` &nbsp;&mdash;&nbsp; `model.py:491`, 24 lines

def _add_cpt_risk(m: Model) -> None:
    """Cumulative demand against cumulative capacity, per carrier cut-off."""
    b, p, xsum = m.backend, m.params, m.backend.xsum
    tol = cfg.THRESHOLDS["cpt_risk"]

    interval = {}
    for c in p.cpts:
        interval[c] = b.num_var(f"H_interval_{c}")
        b.add(
            interval[c]
            == xsum(p.alpha[(c, h)] * m.Hcap[(cfg.OB, h)] for h in p.hours if p.alpha[(c, h)] > 0),
            f"H_interval_{c}",
        )
    for i, c in enumerate(p.cpts):
        m.H_cum[c] = b.num_var(f"H_cum_{c}")
        b.add(
            m.H_cum[c] == xsum(interval[k] for k in p.cpts[: i + 1]),
            f"H_cum_{c}",
        )
        m.g_cpt[c] = b.num_var(f"g_cpt_{c}")
        b.add(
            p.cpt_demand[c] <= tol * m.H_cum[c] + m.g_cpt[c],
            f"cpt_risk_{c}",
        )

_add_cpt_risk         874 vars   706 cons   (+18 vars, +18 cons)


---
### 4.8 Limit 5: the returns ceiling

$$
\begin{aligned}
  C^{\mathrm{rt}}_{h} &= \sum_{k \in \mathcal{R}_{h}} \mathrm{Hcap}_{\mathrm{RT},k},
  \qquad
  U_{\mathrm{RT},h} \le \delta \cdot \frac{24}{|\mathcal{R}_{h}|}\,C^{\mathrm{rt}}_{h}
    + w^{\mathrm{rt}}_{h}
\end{aligned}
$$

Stated in **days** of forward cover ($\delta = 3$) where the outbound band is in hours - the
two are not interchangeable and the denominators differ accordingly. The window is clipped to
the horizon and rescaled by the hours actually present, rather than divided by a constant,
for the same reason as the band.

In [34]:
show_source(model._add_returns_band)
model._add_returns_band(m)
report(m, '_add_returns_band')

`_add_returns_band` &nbsp;&mdash;&nbsp; `model.py:520`, 21 lines

def _add_returns_band(m: Model) -> None:
    """Returns backlog under a days-of-cover ceiling.

    The window is clipped to the horizon and scaled by the hours present, so clipping
    does not tighten the ceiling.
    """
    b, p, xsum = m.backend, m.params, m.backend.xsum
    days = cfg.THRESHOLDS["returns_bl_days"]

    for h in p.hours:
        window = p.rt_window[h]
        m.w_rt[h] = b.num_var(f"w_rt_{int(h)}")
        if not window:
            continue
        m.C_RT[h] = b.num_var(f"C_RT_{int(h)}")
        b.add(
            m.C_RT[h] == xsum(m.Hcap[(cfg.RT, k)] for k in window),
            f"C_RT_{int(h)}",
        )
        daily = m.C_RT[h] * (24.0 / len(window))
        b.add(m.U[(cfg.RT, h)] <= days * daily + m.w_rt[h], f"rt_bl_{int(h)}")

_add_returns_band     922 vars   754 cons   (+48 vars, +48 cons)


---
## 5. The objective ladder

Fifteen ranks, minimised in strict priority order. A lower rank can never trade against a
higher one, however large the numbers.

The three `kind` bands mean different things and must not be summed together:

- **slack** - a constraint violation, read as *units breached*. Zero is healthy.
- **guardrail** - distance from a target sitting strictly inside the operational limit. Never
  zero on a real day, and a non-zero value is **not a defect** - it is risk carried.
- **preference** - taste among already-feasible solutions. A price paid, not a breach.

Rank 15 carries a minus sign, so it *maximises* people parked in Support. Support produces
nothing, which makes it a tie-break rather than a cost - and it is what makes the output
deterministic run to run.

In [35]:
m.tiers = model._build_ladder(m, m.backend.xsum)
report(m, "_build_ladder")

pd.DataFrame(
    [{"rank": i, "tier": t.name, "label": t.label, "kind": t.kind}
     for i, t in enumerate(m.tiers, 1)]
).set_index("rank")

_build_ladder         922 vars   754 cons   (+0 vars, +0 cons)


,tier,label,kind
rank,,,
1,storage_now,Inbound storage (this shift),slack
2,storage_later,Inbound storage (later),slack
3,cpt_now,Carrier cut-off risk (this shift),slack
4,cpt_later,Carrier cut-off risk (later),slack
5,ob_bl_max_now,Outbound backlog too high (this shift),slack
6,ob_bl_max_later,Outbound backlog too high (later),slack
7,eod_max,End-of-day backlog above target,slack
8,ob_bl_min_now,Outbound backlog too low (this shift),slack
9,ob_bl_min_later,Outbound backlog too low (later),slack


---
## 6. Solve it twice

Every published quantity is a **pair**. The *before* solve pins every movement to zero and
must reproduce the planned trajectory exactly; the *after* solve lets the model reallocate.
Without that identity no before/after difference on any chart would be attributable to the
recommendation rather than to drift in how the baseline was reconstructed.

`run.run_scenario` does both in one call - that is what `run_demo.py` uses. The ladder itself
is `02-solve-the-ladder.ipynb`.

In [36]:
result = run.run_scenario(scn, out_dir=OUT)

nv, nc = result.after.backend.report_size()
print(f"after solve: {nv} variables, {nc} constraints, {len(result.ladder)} ranks")
print(f"  the {nc - 754} extra constraints are the freeze added once per rank while solving\n")
print(f"movements recommended: {result.moves_total}")
print(f"limit reported       : {result.limit}")
print(f"resolved             : {result.resolved}")

after solve: 922 variables, 769 constraints, 15 ranks
  the 15 extra constraints are the freeze added once per rank while solving

movements recommended: 25
limit reported       : resolved
resolved             : True


In [37]:
# Published as three CSVs - into the scratch dir, not ../../output.
written, preserved = run.write_data([result], OUT)
for pth in written:
    print(f"  {pth.name:<14} {pth.stat().st_size:>7,} bytes")
print(f"\nkept from a previous run: {sorted(preserved) or 'nothing'}")

  moves.csv          343 bytes
  review.csv       1,616 bytes
  detail.csv      10,730 bytes

kept from a previous run: nothing


---
## 7. Does this agree with the CLI?

The notebook is only worth trusting if it builds the same model `run_demo.py` does. Three
assertions: the model size, the rank count, and the ladder values themselves.

In [38]:
EXPECTED_VARS, EXPECTED_CONS, EXPECTED_RANKS = 922, 769, 15

nv, nc = result.after.backend.report_size()
assert (nv, nc) == (EXPECTED_VARS, EXPECTED_CONS), f"model size moved: {nv}/{nc}"
assert len(result.ladder) == EXPECTED_RANKS, f"rank count moved: {len(result.ladder)}"

# The stage-by-stage build above and run_scenario's own build must agree on the ladder.
staged = {t.name for t in m.tiers}
solved = set(result.ladder["tier"])
assert staged == solved, f"ladder differs: {staged ^ solved}"

print(f"OK  {nv} variables / {nc} constraints / {EXPECTED_RANKS} ranks")
print(f"OK  the staged build and run_scenario produced the same {len(staged)} ranks")
print(f"\nto confirm against the CLI:  cd .. && python run_demo.py --scenario {scn.name} --out /tmp/x")

OK  922 variables / 769 constraints / 15 ranks
OK  the staged build and run_scenario produced the same 15 ranks

to confirm against the CLI:  cd .. && python run_demo.py --scenario baseline --out /tmp/x
